In [2]:
!ls /opt/spark/work-dir/notebooks/labs/data

HEADER.csv			      videogames_20260415_162618_00003.csv
videogames_20260415_162535_00001.csv  videogames_20260415_162639_00004.csv
videogames_20260415_162556_00002.csv  videogames_20260415_162700_00005.csv


In [1]:
from pcamarillor.spark_utils import SparkUtils
from pyspark.sql.functions import col

# 🔥 Spark
su = SparkUtils("CleanVideogames", "local[*]")
spark = su.spark

# 📥 1. Leer TODO (sin confiar en header)
df = spark.read \
    .option("header", "false") \
    .option("inferSchema", "false") \
    .option("mode", "PERMISSIVE") \
    .csv("/opt/spark/work-dir/notebooks/labs/data")

# 🧩 2. Forzar columnas (estructura esperada)
columns = [
    "id", "timestamp", "name", "game", "genre",
    "platform", "country", "price", "quantity", "rating"
]

df = df.toDF(*columns)

# 🔍 DEBUG inicial
print("ANTES DE LIMPIEZA:")
df.show(5)

# 🔥 3. Eliminar filas tipo header mezclado
# (ej: "sale_price_usd", "event_id", etc.)
df = df.filter(~col("price").isin("sale_price", "sale_price_usd", "price"))

# 🔥 4. Cast seguro (los inválidos = null)
df = df.withColumn("price", col("price").cast("double")) \
       .withColumn("rating", col("rating").cast("double")) \
       .withColumn("quantity", col("quantity").cast("int"))

# 🔥 5. Eliminar filas inválidas reales
df_clean = df.filter(
    col("price").isNotNull() &
    col("quantity").isNotNull()
)

# 🔍 DEBUG después
print("DESPUÉS DE LIMPIEZA:")
df_clean.show(5)

# 📊 Opcional: ver nulos
su.count_nulls(df_clean).show()

# 💾 6. Guardar limpio (LOCAL)
df_clean.write \
    .mode("overwrite") \
    .partitionBy("genre") \
    .parquet("/opt/spark/work-dir/notebooks/labs/clean_output")

print("✅ Dataset limpio generado")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/21 17:40:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/21 17:40:25 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


ANTES DE LIMPIEZA:
+--------------------+-------------------+-----------------+--------------------+----------+----------------+--------------------+-----+--------+------+
|                  id|          timestamp|             name|                game|     genre|        platform|             country|price|quantity|rating|
+--------------------+-------------------+-----------------+--------------------+----------+----------------+--------------------+-----+--------+------+
|a0d92fb3-efff-457...|2025-03-13 22:05:45|       SonVoice60|      The Event Much|Simulation| PC (Battle.net)|Libyan Arab Jamah...| 23.5|       1|   3.9|
|a67a4912-61ba-496...|2022-02-14 06:59:20|LimeGreenFaulkner|Help Wars: Past S...|  Fighting|Mobile (Android)|              Uganda|62.85|       1|   9.3|
|112d4512-0831-4c1...|2025-08-23 18:11:53|       juancampos|Present Successfu...|  Strategy|    Mobile (iOS)|United States of ...| 14.2|       3|   6.1|
|d1e5e9a9-7146-4fa...|2023-01-06 14:22:33|         Janet924|  L

+---+---------+----+----+-----+--------+-------+-----+--------+------+
| id|timestamp|name|game|genre|platform|country|price|quantity|rating|
+---+---------+----+----+-----+--------+-------+-----+--------+------+
|  0|        0|   0|   0|    0|       0|      0|    0|       0| 75946|
+---+---------+----+----+-----+--------+-------+-----+--------+------+



26/04/21 17:40:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/21 17:40:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/21 17:40:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/21 17:40:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/21 17:40:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/04/21 17:40:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/21 17:40:36 WARN MemoryManager: Total allocation exceeds 95.

✅ Dataset limpio generado


In [ ]:
!ls

In [2]:
ls /opt/spark/work-dir/notebooks/labs/clean_output

'genre=Action-Adventure'/      'genre=MOBA'/        'genre=Simulation'/
'genre=Battle Royale'/         'genre=Platformer'/  'genre=Sports'/
'genre=Fighting'/              'genre=Puzzle'/      'genre=Strategy'/
'genre=First-Person Shooter'/  'genre=Racing'/       _SUCCESS
'genre=Horror'/                'genre=RPG'/
'genre=MMORPG'/                'genre=Sandbox'/


In [3]:
df_test = spark.read.parquet("/opt/spark/work-dir/notebooks/labs/clean_output")
df_test.show(10)
df_test.printSchema()

+--------------------+-------------------+-----------------+--------------------+----------------+--------------------+-----+--------+------+-------+
|                  id|          timestamp|             name|                game|        platform|             country|price|quantity|rating|  genre|
+--------------------+-------------------+-----------------+--------------------+----------------+--------------------+-----+--------+------+-------+
|834cac46-6a56-4d1...|2020-06-06 01:17:50|        ForSort67|Notice Word Chron...|   PlayStation 5|           Argentina|68.94|       2|   9.9|Sandbox|
|1d12ce9c-27cb-411...|2020-07-15 19:06:19|           eterry|Race Certainly Ch...|   PlayStation 4|              Mexico|30.48|       1|   7.6|Sandbox|
|d62f16f8-791e-438...|2021-07-18 06:23:12|        SeeSong51|Animal of the Cur...| Nintendo Switch|            Zimbabwe|38.43|       1|   4.6|Sandbox|
|bb092252-74fe-45e...|2020-04-08 07:20:12|  pro_rodriguez_3|       Dark Ok: Site|Mobile (Android)|  

In [4]:
df_test.filter(col("price").isNull()).show()

[Stage 11:=================================================>        (6 + 1) / 7]

+---+---------+----+----+--------+-------+-----+--------+------+-----+
| id|timestamp|name|game|platform|country|price|quantity|rating|genre|
+---+---------+----+----+--------+-------+-----+--------+------+-----+
+---+---------+----+----+--------+-------+-----+--------+------+-----+



In [5]:
df_test.groupBy("genre").count().show()

[Stage 12:===================>                                     (4 + 8) / 12]

+--------------------+-----+
|               genre|count|
+--------------------+-----+
|              Horror|63384|
|              Racing|63368|
|              Puzzle|63437|
|                MOBA|63270|
|              MMORPG|63297|
|            Strategy|63271|
|          Simulation|63528|
|             Sandbox|63451|
|              Sports|63505|
|       Battle Royale|63242|
|First-Person Shooter|63295|
|    Action-Adventure|62996|
|            Fighting|63305|
|          Platformer|63328|
|                 RPG|63141|
+--------------------+-----+



26/04/21 19:08:35 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 300517 ms exceeds timeout 120000 ms
26/04/21 19:08:37 WARN SparkContext: Killing executors is not supported by current scheduler.
26/04/21 19:08:36 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.rpc.RpcTimeoutException: Future timed out after [10000 milliseconds]. This timeout is controlled by spark.executor.heartbeatInterval
	at org.apache.spark.rpc.RpcTimeout.org$apache$spark$rpc$RpcTimeout$$createRpcTimeoutException(RpcTimeout.scala:47)
	at org.apache.spark.rpc.RpcTimeout$$anonfun$addMessageIfTimeout$1.applyOrElse(RpcTimeout.scala:62)
	at org.apache.spark.rpc.RpcTimeout$$anonfun$addMessageIfTimeout$1.applyOrElse(RpcTimeout.scala:58)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:35)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:76)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at o